# prepare input txt files

In [1]:
import os
import sys
import pandas as pd
import numpy as np

import json
import pickle

In [2]:
with open("/projects/extern/nhr/nhr_ni/nib00021/dir.project/hamza/repos/bbox_detection_sam3/ssv2_full_bbox_analysis/ssv2_full_bbox_dataframes.pkl", 'rb') as f:
    data = pickle.load(f)
    df_images = data['df_images']

print(df_images.shape)

(193690, 5)


In [3]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

In [4]:
df_images.head(2)

,image_id,height,width,area,num_bboxes
0,projects_extern_nhr_nhr_ni_nib00021_dir.project_data_public_ssv2_clips_78687.jpg,240,428,102720,1
1,projects_extern_nhr_nhr_ni_nib00021_dir.project_data_public_ssv2_clips_42326.jpg,240,427,102480,1


In [ ]:
df_images['id'] = df_images['image_id'].str.split("ssv2_clips_").str[1].str.split('.').str[0]
df_images.head(2)

,image_id,height,width,area,num_bboxes,id
0,projects_extern_nhr_nhr_ni_nib00021_dir.project_data_public_ssv2_clips_78687.jpg,240,428,102720,1,78687
1,projects_extern_nhr_nhr_ni_nib00021_dir.project_data_public_ssv2_clips_42326.jpg,240,427,102480,1,42326


In [11]:
with open('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/public/ssv2/labels/train.json', 'r') as f:
    train_raw = json.load(f)

with open('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/public/ssv2/labels/validation.json', 'r') as f:
    val_raw = json.load(f)

df_train = pd.DataFrame(train_raw)
df_train['split'] = 'train'

df_val = pd.DataFrame(val_raw)
df_val['split'] = 'val'

ssv2_labels_df = pd.concat([df_train, df_val], ignore_index=True)
print(ssv2_labels_df.shape)
ssv2_labels_df.head(2)

(193690, 5)


,id,label,template,placeholders,split
0,78687,holding potato next to vicks vaporub bottle,Holding [something] next to [something],"[potato, vicks vaporub bottle]",train
1,42326,spreading margarine onto bread,Spreading [something] onto [something],"[margarine, bread]",train


In [ ]:
for col in ssv2_labels_df.columns:
    print(f"{col:14}: {ssv2_labels_df[col].astype(str).nunique()}")

id            : 193690
label         : 123858
template      : 174
placeholders  : 65256
split         : 2


In [12]:
df = pd.merge(df_images, ssv2_labels_df[['id','template','split']], how = 'left', on = ['id']) # LEFT JOIN
print(df.shape)
df.head(3)

(193690, 8)


,image_id,height,width,area,num_bboxes,id,template,split
0,projects_extern_nhr_nhr_ni_nib00021_dir.project_data_public_ssv2_clips_78687.jpg,240,428,102720,1,78687,Holding [something] next to [something],train
1,projects_extern_nhr_nhr_ni_nib00021_dir.project_data_public_ssv2_clips_42326.jpg,240,427,102480,1,42326,Spreading [something] onto [something],train
2,projects_extern_nhr_nhr_ni_nib00021_dir.project_data_public_ssv2_clips_100904.jpg,240,427,102480,0,100904,Putting [something] on a surface,train


In [ ]:
# If we drop the 0 bbox examples, will it mess up the train and val split percentages? 
# Answer is No
print(df.split.value_counts(1))
print(df[df.num_bboxes>0].split.value_counts(1))

split
train    0.872079
val      0.127921
Name: proportion, dtype: float64
split
train    0.865353
val      0.134647
Name: proportion, dtype: float64


In [16]:
print(df.template.nunique())
print(df[df.num_bboxes>0].template.nunique())

174
174


In [17]:
df = df[df.num_bboxes>0]
df.shape

(150289, 8)

In [ ]:
# 1. Create a global template-to-integer (actionclass-to-integer) mapping using the full dataset (train + val)
unique_templates = sorted(df["template"].unique())
template2id = {template: idx for idx, template in enumerate(unique_templates)}

# Apply the mapping and format clip identifiers
df["label"] = df["template"].map(template2id)
df["clip"] = df["id"].astype(str) + ".webm"

# 2. Split dataframe and export train.txt and val.txt separately
train_df = df[df["split"] == "train"]
val_df = df[df["split"] == "val"]

train_df[["clip", "label"]].to_csv("data/train.txt", sep=" ", header=False, index=False)
val_df[["clip", "label"]].to_csv("data/val.txt", sep=" ", header=False, index=False)

# 3. Save the single label mapping file for reference
mapping_df = pd.DataFrame(list(template2id.items()), columns=["template", "integer_id"])
mapping_df[["integer_id", "template"]].to_csv(
    "data/template2id_mapping.txt", sep="\t", header=False, index=False
)

# Run extract.py


In [ ]:
"""
Step 0 is to download & cache model on the Login Node
python download_cache_dinov2_model.py
ls ~/.cache/huggingface/hub/

# Run the extraction test: 48 train and 14 val examples in these testing files.
python extract.py \
    --annotations train:data/train_for_testing.txt val:data/val_for_testing.txt \
    --image_dir $COLD/felixbm/ssv2_center_frame_occluding_objects/frames/ \
    --out data/extracted_features/ssv2-dinov2-large-center-testing.h5 \
    --batch_size 12 \
    --num_workers 8

"""

In [1]:
import h5py

h5_path = "data/extracted_features/ssv2-dinov2-large-center-testing.h5"

with h5py.File(h5_path, "r") as f:
    print("Splits stored:", list(f.keys()))
    
    for split in f.keys():
        print(f"\n--- Split: {split} ---")
        group = f[split]
        
        features = group["features"]
        labels = group["labels"]
        names = group["names"]
        
        print(f"  features shape: {features.shape} | dtype: {features.dtype}")
        print(f"  labels shape:   {labels.shape}   | dtype: {labels.dtype}")
        print(f"  names shape:    {names.shape}    | dtype: {names.dtype}")
        print(f"  num_written:    {group.attrs.get('num_written')}")
        
        # Sample inspection
        sample_name = names[0].decode("utf-8") if isinstance(names[0], bytes) else names[0]
        print(f"  Sample 0 -> Name: {sample_name}, Label: {labels[0]}")

Splits stored: ['train', 'val']

--- Split: train ---
  features shape: (48, 1, 256, 1024) | dtype: float16
  labels shape:   (48,)   | dtype: int16
  names shape:    (48,)    | dtype: object
  num_written:    48
  Sample 0 -> Name: 78687, Label: 19

--- Split: val ---
  features shape: (14, 1, 256, 1024) | dtype: float16
  labels shape:   (14,)   | dtype: int16
  names shape:    (14,)    | dtype: object
  num_written:    14
  Sample 0 -> Name: 74225, Label: 135


**Data Structure Walkthrough**

* **Split Groups (`/<split>/`):** Data is indexed inside top-level groups named after your splits (e.g., `/train` and `/val`).


* **`features` (`[N, 1, 256, 1024]`):**
* `N`: Total clips extracted.
* `1`: Number of extracted frames per clip (1 center frame).
* `256`: Spatial patch tokens ($16 \times 16$ grid derived from $224 \times 224$ images at patch size 14).
* `1024`: Feature dimension output by `vit_large` DINOv2.
* **`labels` (`[N]`):** 1D array of integer class targets.
* **`names` (`[N]`):** String array matching clip IDs.



---

**How `train.py` Consumes It**

When you pass this file to `train.py`:

1. `FeatureDataset` inspects `features.shape[1:]` to infer `feat_shape = (1, 256, 1024)` (`[T, P, D]`) and checks spatial dimensions ($16 \times 16 = 256$).


2. PyTorch `DataLoader` streams sample slices `[1, 256, 1024]` lazily by row index.


3. `AttentiveProbe` flattens the frame and spatial patch dimensions (`1 * 256 = 256` tokens), adds 3D sinusoidal positional embeddings, and queries them with cross-attention to output a single prediction vector.



To start training, pass the output path via `--h5`:

```bash
python train.py --h5 data/extracted_features/ssv2-dinov2-large-center-testing.h5 --run_name baseline_test

```

In [ ]:
"""
# Run the full extraction.
python extract.py \
    --annotations train:data/train.txt val:data/val.txt \
    --image_dir $COLD/felixbm/ssv2_center_frame_occluding_objects/frames/ \
    --out data/extracted_features/ssv2-center-dinov2-unoccluded.h5 \
    --batch_size 32 \
    --num_workers 14

"""

In [ ]:
"""
#### OCCLUSION
python extract.py \
    --annotations train:data/train.txt val:data/val.txt \
    --image_dir $COLD/felixbm/ssv2_center_frame_occluding_objects/frames/ \
    --out data/extracted_features/ssv2-center-dinov2-bbox-occluded.h5 \
    --coco_json data/annotations_score0.2_nms0.15.coco.json \
    --occlusion \
    --batch_size 256 \
    --num_workers 20

"""

# train.py

In [ ]:
"""
python train.py \
    --h5 data/extracted_features/ssv2-center-dinov2-unoccluded.h5 \
    --run_name baseline_test \
    --epochs 15 \
    --lr 0.0001 \
    --batch_size 64 \
    --num_workers 14 

# optimizing hyperparams for more efficient training runs
# Increasing batch size drastically, num workers slightly and 
# only increasing lr very slightly (lr should be increased linearly with batchsize but i wanted the 
# effect to be tinier so i only doubled lr with 8x batchsize)
python train.py \
    --h5 data/extracted_features/ssv2-center-dinov2-unoccluded.h5 \
    --run_name baseline_test_2 \
    --epochs 25 \
    --lr 0.0002 \
    --batch_size 128 \
    --num_workers 15

"""

In [ ]:
"""
## OCCLUSION
python train.py \
    --h5 data/extracted_features/ssv2-center-dinov2-bbox-occluded.h5 \
    --run_name bbox_occlusion \
    --epochs 25 \
    --lr 0.0002 \
    --batch_size 128 \
    --num_workers 15

"""